# Notebook 03 — Avaliação do modelo (Etapa 3)

Compara o modelo **base** (Stable Diffusion v1-5) com o **base + LoRA**: varredura de escala,
grade comparativa 6×2, **CLIPScore**, **verificação de memorização** e **avaliação humana cega**.
Inclui o **VAE-fix** (`sd-vae-ft-mse`) que estabiliza o fp16 e reduz deformações. Requer GPU (Runtime → T4).

### 0. Ambiente e configuração

Duas células: **(a) instalação** (roda 1×, instala deps) e **(b) imports + configuração** (monta o Drive).
O dataset (imagens + `metadata.jsonl`) é o mesmo do notebook 01, lido do `DATA_DIR` no Drive — **sem re-baixar**.

> Se após a instalação aparecer erro de import do `torch` (*partially initialized module*),
> **reinicie a sessão** (Ambiente de execução → Reiniciar sessão) e rode a partir da célula (b).

In [ ]:
# (a) INSTALAÇÃO — rode 1x.
# O Colab traz torchao 0.10.0, que o peft (load_lora_weights) REJEITA -> desinstalamos.
!pip uninstall -y torchao
!pip -q install "diffusers" "transformers" "accelerate" "peft"
print(">> Deps instaladas. Se a próxima célula der erro de import do torch,")
print(">> reinicie a sessão e rode a partir dela.")

In [ ]:
# (b) IMPORTS + CONFIGURAÇÃO (sobrevive a reinício da sessão).
import os, json, random, glob
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive, userdata
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Ative a GPU: Runtime -> Alterar tipo de ambiente -> T4 GPU."
dtype = torch.float16
print("GPU:", torch.cuda.get_device_name(0))

# --- Modelos ---
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_REPO  = "lamble-lambe/atelie"     # oficial da equipe (usado no app). Alt.: "higosoares/atelie-estilo-lambelambe"
LORA_REVISION = "v1.0.0"               # tag no Hub (ex.: v1.0.0, v1.0.1); "" = main

# --- Parâmetros de geração (refinados) ---
SEED = 42
STEPS = 40
GUIDANCE = 7.0
LORA_SCALE = 0.7                        # intensidade do LoRA (ajuste pela varredura da seção 3)
STYLE_TOKEN = "estilo_lambelambe,"
NEG_PROMPT = ("color, colorful, modern, blurry, out of focus, deformed, disfigured, deformed face, "
              "extra fingers, extra limbs, mutated hands, bad anatomy, ugly, low quality, "
              "low resolution, jpeg artifacts, watermark, text")

PROMPTS = [
    "estilo_lambelambe, a close vintage portrait of an elderly man with a hat, detailed face, sharp focus, black and white, fine grain",
    "estilo_lambelambe, a portrait of a young woman in a lace dress, cabinet card, detailed face, studio lighting, black and white, fine grain",
    "estilo_lambelambe, a portrait of a street lambe-lambe photographer with a wooden box camera, sepia, detailed, sharp focus, old photograph",
    "estilo_lambelambe, a portrait of a seated soldier looking at the camera, tintype, detailed face, sharp focus, aged photograph",
    "estilo_lambelambe, a family portrait, people posing close together, vintage, detailed faces, classic studio composition, black and white",
    "estilo_lambelambe, a close portrait of a child, detailed face, sharp focus, grainy, early 20th century",
]

# --- Caminhos no Drive (MESMO dataset dos notebooks 01/02) ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/multimodais/dados"       # imagens + metadata.jsonl (notebook 01)
OUT_DIR  = "/content/drive/MyDrive/Colab Notebooks/multimodais/avaliacao"   # saídas da avaliação
os.makedirs(OUT_DIR, exist_ok=True)

try:
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("Aviso login HF (repos públicos funcionam sem token):", e)

print("Avaliando:", LORA_REPO, "@", LORA_REVISION or "main", "| dataset:", DATA_DIR)

### 1. Carregar os modelos (SD + VAE-fix + LoRA + CLIP)

O **VAE-fix** (`stabilityai/sd-vae-ft-mse`) substitui o VAE original, instável em fp16, e **reduz
deformações**. O LoRA é carregado **uma vez**; a intensidade é controlada por `scale` na função `gerar`.

> O aviso *"No LoRA keys associated to CLIPTextModel ... prefix='text_encoder'"* é **normal** (LoRA só no UNet).

In [ ]:
from diffusers import StableDiffusionPipeline, AutoencoderKL
from transformers import CLIPModel, CLIPProcessor

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, safety_checker=None
).to(device)
pipe.set_progress_bar_config(disable=True)

# VAE estável em fp16 (o VAE original do SD1.5 causa manchas/deformações em meia-precisão)
pipe.vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=dtype).to(device)

# LoRA carregado UMA vez; base = scale 0, LoRA = scale LORA_SCALE (ver função gerar)
pipe.load_lora_weights(LORA_REPO, revision=LORA_REVISION or None)

# CLIP (CLIPScore + memorização)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

print(f"Modelos prontos | LoRA {LORA_REPO}@{LORA_REVISION or 'main'} | VAE sd-vae-ft-mse")

### 2. Funções auxiliares

`gerar(prompt, seed, usar_lora, lora_scale)` — base (scale 0) ou LoRA (scale). Mesma seed = comparação justa.
`clip_score(img, texto)` — CLIPScore manual (0–100). `embed(img)` — embedding CLIP para memorização.

In [ ]:
def gerar(prompt, seed=SEED, usar_lora=True, lora_scale=LORA_SCALE):
    g = torch.Generator(device=device).manual_seed(int(seed))
    escala = float(lora_scale) if usar_lora else 0.0
    return pipe(prompt, negative_prompt=NEG_PROMPT, guidance_scale=GUIDANCE,
                num_inference_steps=STEPS, generator=g,
                cross_attention_kwargs={"scale": escala}).images[0]

def clip_score(img, texto):
    """CLIPScore = 100 * cos(embedding_imagem, embedding_texto). Maior = mais coerente com o texto."""
    ins = clip_processor(text=[texto], images=img, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = clip_model(**ins)
    ie = F.normalize(out.image_embeds, p=2, dim=-1)
    te = F.normalize(out.text_embeds, p=2, dim=-1)
    return (100.0 * (ie * te).sum(dim=-1)).item()

def embed(img):
    ins = clip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        return F.normalize(clip_model.get_image_features(**ins), p=2, dim=-1)

### 3. Varredura de intensidade do LoRA (scale)

Gera o mesmo prompt (mesma seed) em várias escalas para escolher o melhor equilíbrio **estilo × rosto**.
Ajuste `LORA_SCALE` na célula (b) com o valor escolhido antes de rodar as próximas seções.

In [ ]:
scales = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
prompt_teste = PROMPTS[0]

fig, axes = plt.subplots(1, len(scales) + 1, figsize=(4 * (len(scales) + 1), 4.5))
axes[0].imshow(gerar(prompt_teste, SEED, usar_lora=False)); axes[0].set_title("Base"); axes[0].axis("off")
for i, s in enumerate(scales):
    axes[i + 1].imshow(gerar(prompt_teste, SEED, usar_lora=True, lora_scale=s))
    axes[i + 1].set_title(f"scale={s}"); axes[i + 1].axis("off")
fig.suptitle("Varredura de escala do LoRA", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"varredura_scale_{LORA_REVISION or 'main'}.png"), dpi=130, bbox_inches="tight")
plt.show()

### 4. Grade comparativa base × LoRA (6 prompts, mesma seed)

Gera os 6 prompts com base e LoRA usando a **mesma seed** e monta a grade 6×2 rotulada (salva em `saidas_avaliacao/`).

In [ ]:
base_imgs, lora_imgs = [], []
for i, p in enumerate(PROMPTS, 1):
    print(f"[{i}/{len(PROMPTS)}] {p[:55]}...")
    base_imgs.append(gerar(p, SEED, usar_lora=False))
    lora_imgs.append(gerar(p, SEED, usar_lora=True))

n = len(PROMPTS)
fig, axes = plt.subplots(n, 2, figsize=(9, 4.4 * n))
for r in range(n):
    axes[r, 0].imshow(base_imgs[r]); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    axes[r, 1].imshow(lora_imgs[r]); axes[r, 1].axis("off")
    axes[r, 0].set_ylabel(f"{r+1}", rotation=0, labelpad=14, fontsize=12, va="center")
axes[0, 0].set_title("Base (SD 1.5)", fontsize=14, color="#333333")
axes[0, 1].set_title(f"Base + LoRA (scale={LORA_SCALE})", fontsize=14, color="#56148a")
fig.suptitle(f"Grade comparativa — {LORA_REPO} @ {LORA_REVISION or 'main'}", fontsize=15)
plt.tight_layout()
grade_path = os.path.join(OUT_DIR, f"grade_comparativa_{LORA_REVISION or 'main'}.png")
plt.savefig(grade_path, dpi=120, bbox_inches="tight")
plt.show()
print("Grade salva em:", grade_path)
for i, p in enumerate(PROMPTS, 1):
    print(f"  {i}. {p}")

### 5. CLIPScore (coerência imagem–texto): base × LoRA

In [ ]:
scores_base = [clip_score(base_imgs[i], PROMPTS[i]) for i in range(len(PROMPTS))]
scores_lora = [clip_score(lora_imgs[i], PROMPTS[i]) for i in range(len(PROMPTS))]

print(f"CLIPScore médio — Base: {np.mean(scores_base):.2f}")
print(f"CLIPScore médio — LoRA: {np.mean(scores_lora):.2f}")

df_clip = pd.DataFrame({
    "prompt": [f"Prompt {i+1}" for i in range(len(PROMPTS))],
    "clip_base": [round(s, 2) for s in scores_base],
    "clip_lora": [round(s, 2) for s in scores_lora],
})
df_clip.to_csv(os.path.join(OUT_DIR, "clip_scores.csv"), index=False)
df_clip

### 6. Verificação de memorização (overfitting)

Usa as **imagens de treino já no Drive** (do notebook 01 — sem re-baixar). Gera 10 imagens com legendas
próximas às do treino e mede a **similaridade máxima** (CLIP) com o conjunto de treino.
Similaridade muito alta (> 0.95) sugere que o modelo **copiou** uma imagem de treino.

In [ ]:
# imagens de treino já estão no Drive (produzidas pelo notebook 01) — sem re-baixar
treino_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpg")) + glob.glob(os.path.join(DATA_DIR, "*.png")))
assert treino_paths, f"Nenhuma imagem em {DATA_DIR}. Rode o notebook 01 (ou confira o caminho)."
treino_imgs = [Image.open(p).convert("RGB") for p in treino_paths]
emb_treino = torch.cat([embed(im) for im in treino_imgs], dim=0)   # (N, D)
print(f"{len(treino_imgs)} imagens de treino carregadas de {DATA_DIR}")

# legendas do metadata.jsonl (mesmo do treino) -> amostra 10
legendas = [json.loads(l)["text"] for l in open(os.path.join(DATA_DIR, "metadata.jsonl"), encoding="utf-8")]
random.seed(0)
amostra = random.sample(legendas, min(10, len(legendas)))

sims = []
for i, cap in enumerate(amostra, 1):
    ger = gerar(cap, seed=100 + i, usar_lora=True)
    sim = (embed(ger) @ emb_treino.T).max().item()   # similaridade máxima com o treino
    sims.append(sim)
    print(f"[{i:02d}] sim_max={sim:.3f} | {cap[:55]}")

print(f"\nSimilaridade média com o treino: {np.mean(sims):.3f}")
print("Regra prática: sim_max > 0.95 sugere MEMORIZAÇÃO. Suspeitos:", sum(s > 0.95 for s in sims), "de", len(sims))

### 7. Avaliação humana (formulário cego)

Embaralha as imagens base e LoRA, salva com nomes anônimos e gera `formulario.csv` para
**≥ 5 avaliadores** darem nota 1–5 de **fidelidade ao estilo** e **qualidade geral** sem saber a origem.
O gabarito (`_gabarito.csv`) fica oculto — é a chave base/LoRA.

In [ ]:
import csv
HUMAN_DIR = os.path.join(OUT_DIR, "avaliacao_humana")
os.makedirs(HUMAN_DIR, exist_ok=True)

itens = []
for i, (b, l) in enumerate(zip(base_imgs, lora_imgs)):
    itens.append(("base", i, b)); itens.append(("lora", i, l))
random.seed(7); random.shuffle(itens)

chave = []
for k, (modelo, idx, img) in enumerate(itens, 1):
    nome = f"img_{k:02d}.png"
    img.save(os.path.join(HUMAN_DIR, nome))
    chave.append({"arquivo": nome, "modelo": modelo, "prompt": PROMPTS[idx]})

pd.DataFrame(chave).to_csv(os.path.join(HUMAN_DIR, "_gabarito.csv"), index=False)          # NÃO enviar
pd.DataFrame({"avaliador": "", "arquivo": [c["arquivo"] for c in chave],
             "fidelidade_estilo_1a5": "", "qualidade_geral_1a5": ""}).to_csv(
             os.path.join(HUMAN_DIR, "formulario.csv"), index=False)

print("Pasta para avaliação cega:", HUMAN_DIR)
print(f"- Envie img_01..img_{len(itens):02d}.png + formulario.csv para >= 5 pessoas.")
print("- NAO compartilhe _gabarito.csv (chave base/LoRA).")

### 8. Consolidar a avaliação humana (após coletar as respostas)

Empilhe as respostas em `avaliacao_humana/respostas.csv` (colunas do formulário) e rode para ver as médias.

In [ ]:
import csv
gpath = os.path.join(HUMAN_DIR, "_gabarito.csv")
rpath = os.path.join(HUMAN_DIR, "respostas.csv")
gab = {r["arquivo"]: r["modelo"] for _, r in pd.read_csv(gpath).iterrows()}
if os.path.exists(rpath):
    df = pd.read_csv(rpath)
    df["modelo"] = df["arquivo"].map(gab)
    print("Avaliação humana cega (1-5):")
    print(df.groupby("modelo")[["fidelidade_estilo_1a5", "qualidade_geral_1a5"]].mean().round(2))
else:
    print("Coloque respostas.csv em", HUMAN_DIR, "e rode novamente.")

### Resumo para o relatório (Critério 4)

- **Varredura de escala** e a escolha de `LORA_SCALE`.
- **Grade 6×2** base × LoRA (`grade_comparativa_*.png`).
- **CLIPScore médio** (base × LoRA) — `clip_scores.csv`.
- **Memorização**: similaridade média e suspeitos (>0.95) — conclua sobre overfitting.
- **Avaliação humana** cega (≥5 pessoas): médias de fidelidade e qualidade.
- **Conclusões honestas** (limitações: rostos pequenos, cenas de ação).

> Repita com outra `LORA_REVISION` (Config 1 × Config 2) para embasar o Critério 3.
> Melhoria-chave desta versão: **VAE-fix** (sd-vae-ft-mse) reduz deformações do SD-1.5 em fp16.